# EAT + ABMIL on TCGA NSCLC

A minimal demo showing that **Eliminating Ambiguity Tile-filtering (EAT)** improves a
vanilla **ABMIL** baseline on NSCLC (download feature data from https://huggingface.co/datasets/MahmoodLab/UNI2-h-features).

Pipeline:
1. Train ABMIL on slide-level UNI features (one H5 per slide). To reproduce the same pipeline of this paper, you can refer to truecam/diverse_mil/train_mil.py.
2. Train a tile-level **LightGBM** proxy on a small sub-sample of training tiles. (We use *lightgbm* for fast training and *autogluon* may show better performance due to ensemble.)
3. Use the proxy's per-tile entropy as an *ambiguity* score and find the best
   discard threshold on the validation set (`binary_search_for_best_threshold`).
4. Re-evaluate ABMIL on the test set with high-ambiguity tiles masked out.

EAT helpers live in [`truecam/eat_utils.py`](../truecam/eat_utils.py); the
ABMIL backbone is reused from [`truecam.diverse_mil.downstream_models.abmil`](../truecam/diverse_mil/downstream_models/abmil.py).

In [1]:
"""EAT + ABMIL demo on TCGA NSCLC.

This notebook is a minimal end-to-end showcase that EAT-based tile filtering
improves a vanilla ABMIL baseline on the LUAD vs. LUSC subtyping task.
Most logic lives in the Python package; the notebook only orchestrates it.
"""
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/user/wangtao/TRUECAM")
sys.path.insert(0, str(PROJECT_ROOT))

import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import lightgbm as lgb
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from truecam.diverse_mil.downstream_models.abmil import GatedABMIL
from truecam.diverse_mil.utils import seed_everything
from truecam.eat_utils import (
    create_ambiguity_dict,
    binary_search_for_best_threshold,
    evaluate_eat_patient_metric,
    evaluate_tile_average_accuracy,
)

# --- Configuration ---------------------------------------------------------
H5_FOLDER = Path("/home/user/sngp/uni2-h-features/TCGA")  # contains TCGA-LUAD/, TCGA-LUSC/
CSV_PATH = PROJECT_ROOT / "dataset_csv/nsclc/nsclc_labels_one_run.csv"
SPLIT_COL = "split0"
LABEL_DICT = {"LUAD": 0, "LUSC": 1}
N_CLASSES = len(LABEL_DICT)
EPOCHS = 20
LR = 1e-4
NUM_WORKERS = 4
SEED = 2024
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

seed_everything(SEED)
print(f"Device: {DEVICE}")

/home/user/.conda/envs/gigapath/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Device: cuda


In [2]:
df = pd.read_csv(CSV_PATH)
slide_to_patient = dict(zip(df["slide"], df["patient"]))
df["label"] = df["cohort"].map(LABEL_DICT)

splits = {s: df[df[SPLIT_COL] == s] for s in ["train", "val", "test"]}
slide_label = {s: dict(zip(splits[s]["slide"], splits[s]["label"])) for s in splits}

for s, sub in splits.items():
    print(f"{s:5s}: {len(sub):4d} slides | {sub['patient'].nunique():4d} patients | "
          f"class counts={sub['cohort'].value_counts().to_dict()}")

train:  603 slides |  603 patients | class counts={'LUAD': 328, 'LUSC': 275}
val  :  144 slides |  144 patients | class counts={'LUSC': 87, 'LUAD': 57}
test :  194 slides |  194 patients | class counts={'LUSC': 112, 'LUAD': 82}


## 1. Bag dataset and ABMIL training

In [3]:
class NSCLCBagDataset(Dataset):
    """One H5 file per slide → one bag of tile features.

    If ``ambiguity_dict`` and ``threshold`` are provided, tiles whose ambiguity
    score is above the threshold are dropped (EAT masking). The ambiguity
    array length must match the slide's full tile count, so the proxy must be
    run with no tile sampling on val/test (see next cell).
    """

    def __init__(self, h5_folder, slide_label_map, slide_patient_map,
                 ambiguity_dict=None, threshold=None):
        self.paths, self.labels, self.patients = [], [], []
        for sub in ("TCGA-LUAD", "TCGA-LUSC"):
            for p in (h5_folder / sub).glob("*.h5"):
                if p.stem in slide_label_map:
                    self.paths.append(p)
                    self.labels.append(slide_label_map[p.stem])
                    self.patients.append(slide_patient_map.get(p.stem, p.stem))
        assert self.paths, f"No H5 files found in {h5_folder}"
        self.ambiguity_dict = ambiguity_dict
        self.threshold = threshold

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with h5py.File(self.paths[idx], "r") as f:
            features = f["features"][:].astype(np.float32)
        if features.ndim == 3:        # (1, N, D) → (N, D)
            features = features[0]
        if self.ambiguity_dict is not None:
            amb = self.ambiguity_dict.get(self.paths[idx].stem)
            if amb is not None and len(amb) == len(features):
                features = features[amb < self.threshold]
        return {
            "features": torch.from_numpy(features),
            "label": self.labels[idx],
            "patient": self.patients[idx],
        }


def collate_fn(batch):
    return {
        "features": torch.stack([b["features"] for b in batch]),
        "labels": torch.tensor([b["label"] for b in batch]),
        "patient": [b["patient"] for b in batch],
    }


# Infer feature dim from a sample slide
sample = next((H5_FOLDER / "TCGA-LUAD").glob("*.h5"))
with h5py.File(sample, "r") as f:
    EMBED_DIM = f["features"].shape[-1]
print(f"Embedding dim = {EMBED_DIM}")

Embedding dim = 1536


In [4]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score


def evaluate_abmil(model, loader, tag="eval"):
    """Slide-level inference + patient-level mean aggregation → (acc, bacc, auroc, probs, labels)."""
    model.eval()
    rows = []
    with torch.inference_mode():
        for batch in tqdm(loader, desc=f"eval[{tag}]", leave=False):
            logits = model(batch["features"].float().to(DEVICE)).cpu().numpy()
            for i, p in enumerate(batch["patient"]):
                rows.append((p, int(batch["labels"][i]), *logits[i]))
    cols = ["patient", "label"] + [f"logit_{c}" for c in range(N_CLASSES)]
    df = pd.DataFrame(rows, columns=cols)
    agg = df.groupby("patient").agg(
        {**{c: "mean" for c in cols[2:]}, "label": "first"}
    ).reset_index()
    probs = torch.softmax(torch.tensor(agg[cols[2:]].values), dim=1).numpy()
    y, y_hat = agg["label"].values, probs.argmax(axis=1)
    return (accuracy_score(y, y_hat),
            balanced_accuracy_score(y, y_hat),
            roc_auc_score(y, probs[:, 1]) if N_CLASSES == 2 else float("nan"),
            probs, y)


def train_abmil(train_loader, val_loader, embed_dim, epochs=EPOCHS, lr=LR):
    model = GatedABMIL(embed_dim=embed_dim, n_classes=N_CLASSES).to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=len(train_loader) * epochs)
    criterion = nn.CrossEntropyLoss()
    best_acc, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in tqdm(train_loader, desc=f"epoch {epoch}/{epochs}", leave=False):
            opt.zero_grad()
            logits = model(batch["features"].float().to(DEVICE))
            loss = criterion(logits, batch["labels"].long().to(DEVICE))
            loss.backward()
            opt.step()
            sched.step()
        val_acc, *_ = evaluate_abmil(model, val_loader, tag="val")
        print(f"epoch {epoch}: val acc = {val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_acc


train_loader = DataLoader(NSCLCBagDataset(H5_FOLDER, slide_label["train"], slide_to_patient),
                          batch_size=1, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(NSCLCBagDataset(H5_FOLDER, slide_label["val"], slide_to_patient),
                        batch_size=1, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
test_loader = DataLoader(NSCLCBagDataset(H5_FOLDER, slide_label["test"], slide_to_patient),
                         batch_size=1, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

abmil_model, best_val_acc = train_abmil(train_loader, val_loader, EMBED_DIM)
base_acc, base_bacc, base_auc, _, _ = evaluate_abmil(abmil_model, test_loader, tag="test")
print(f"\nBaseline ABMIL — val={best_val_acc:.4f} | test acc={base_acc:.4f} bacc={base_bacc:.4f} auroc={base_auc:.4f}")

epoch 1: val acc = 0.9236


epoch 2: val acc = 0.9167


epoch 3: val acc = 0.9306


epoch 4: val acc = 0.9375


epoch 5: val acc = 0.9375


...

epoch 17: val acc = 0.9306


epoch 18: val acc = 0.9306


epoch 19: val acc = 0.9306


epoch 20: val acc = 0.9306



Baseline ABMIL — val=0.9444 | test acc=0.9381 bacc=0.9432 auroc=0.9631


## 2. Tile-level LightGBM proxy

Train on a small sub-sample of training tiles (we only use **20%** data (For NSCLC, we prefer 40% percent data to train proxy model for all models), try larger percent and autogluon to unlock the performance); score **all** tiles in the
validation and test slides — keeping the full set is required so the per-tile
ambiguity vector aligns 1-to-1 with the bag dataset.

In [6]:
def build_tile_dataset(slide_label_map, sample_fraction=1.0):
    """Concatenate tile features across slides. ``sample_fraction`` < 1 only valid
    for the training set — val/test must keep all tiles so the per-slide
    ambiguity vector aligns with the bag dataset's tile order."""
    X, y, tags = [], [], []
    for sub in ("TCGA-LUAD", "TCGA-LUSC"):
        for p in tqdm(list((H5_FOLDER / sub).glob("*.h5")), desc=sub, leave=False):
            if p.stem not in slide_label_map:
                continue
            with h5py.File(p, "r") as f:
                feats = f["features"][:].astype(np.float32)
            if feats.ndim == 3:
                feats = feats[0]
            if sample_fraction < 1.0:
                k = max(1, int(len(feats) * sample_fraction))
                feats = feats[np.random.choice(len(feats), k, replace=False)]
            X.append(feats)
            y.append(np.full(len(feats), slide_label_map[p.stem], dtype=np.int8))
            tags.append(np.full(len(feats), p.stem))
    return np.concatenate(X), np.concatenate(y), np.concatenate(tags)


print("Building tile datasets …")
Xtr, ytr, _      = build_tile_dataset(slide_label["train"], sample_fraction=0.2)
Xval, yval, tval = build_tile_dataset(slide_label["val"],   sample_fraction=1.0)
Xte,  yte,  tte  = build_tile_dataset(slide_label["test"],  sample_fraction=1.0)
print(f"train={Xtr.shape} | val={Xval.shape} | test={Xte.shape}")

proxy = lgb.LGBMClassifier(
    boosting_type="gbdt", num_leaves=64, learning_rate=0.02, n_estimators=400,
    objective="binary", class_weight="balanced", subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, device='cpu',
    random_state=SEED, n_jobs=16, verbosity=-1,
)
proxy.fit(Xtr, ytr, eval_set=[(Xval, yval)],
          eval_metric=["binary_logloss", "binary_error"],
          callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])

val_probs  = proxy.predict_proba(Xval)
test_probs_proxy = proxy.predict_proba(Xte)
print(f"proxy val tile acc={accuracy_score(yval, val_probs.argmax(1)):.4f}")

Building tile datasets …


train=(1325975, 1536) | val=(1606826, 1536) | test=(2225555, 1536)
proxy val tile acc=0.7743


## 3. EAT-masked ABMIL evaluation

In [8]:
# Build ambiguity dicts for val and test tiles (needed for EAT masking in ABMIL)
val_amb,  val_quantiles = create_ambiguity_dict(val_probs,  tval)
test_amb, _             = create_ambiguity_dict(test_probs_proxy, tte)
ambiguity_dict = {**val_amb, **test_amb}


def evaluate_abmil_at_threshold(thresh, abmil_model, ambiguity_dict, slide_label, slide_to_patient,
                                val_loader_ref, test_loader_ref):
    """Run ABMIL inference on val and test sets with tile masking at `thresh`.
    Returns (val_acc, val_bacc, val_auc, test_acc, test_bacc, test_auc)."""
    from torch.utils.data import DataLoader
    results = {}
    for split, loader_ref in [("val", val_loader_ref), ("test", test_loader_ref)]:
        eat_loader = DataLoader(
            NSCLCBagDataset(H5_FOLDER, slide_label[split], slide_to_patient,
                            ambiguity_dict=ambiguity_dict, threshold=thresh),
            batch_size=1, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
        acc, bacc, auc, _, _ = evaluate_abmil(abmil_model, eat_loader, tag=f"{split}+EAT@{thresh:.3f}")
        results[split] = (acc, bacc, auc)
    return (*results["val"], *results["test"])

thresholds = np.linspace(val_quantiles[0], val_quantiles[-1], 10)

rows = []
for thresh in tqdm(thresholds, desc="Grid-search EAT thresholds"):
    val_acc, val_bacc, val_auc, test_acc, test_bacc, test_auc = \
        evaluate_abmil_at_threshold(thresh, abmil_model, ambiguity_dict,
                                   slide_label, slide_to_patient,
                                   val_loader, test_loader)
    rows.append({
        "Threshold":   f"{thresh:.4f}",
        "Val Acc":     val_acc,
        "Val BAcc":    val_bacc,
        "Val AUROC":   val_auc,
        "Test Acc":    test_acc,
        "Test BAcc":   test_bacc,
        "Test AUROC":  test_auc,
    })

# Baseline ABMIL (no masking)
rows.append({
    "Threshold":   "baseline",
    "Val Acc":     best_val_acc,          # stored from train_abmil()
    "Val BAcc":    float("nan"),
    "Val AUROC":   float("nan"),
    "Test Acc":    base_acc,
    "Test BAcc":   base_bacc,
    "Test AUROC":  base_auc,
})

result_df = pd.DataFrame(rows).set_index("Threshold")
display(result_df.round(4))

# Also show best test row by each metric
print("\nBest by Val BAcc:", result_df.iloc[result_df["Val BAcc"].argmax()].name)
print("Best by Test BAcc:", result_df.iloc[result_df["Test BAcc"].argmax()].name)


Grid-search EAT thresholds: 100%|██████████| 10/10 [01:56<00:00, 11.62s/it]


,Val Acc,Val BAcc,Val AUROC,Test Acc,Test BAcc,Test AUROC
Threshold,,,,,,
0.0325,0.9028,0.8802,0.9651,0.9175,0.9106,0.9657
0.0785,0.9306,0.9244,0.9706,0.9278,0.9261,0.9613
0.1246,0.9306,0.9244,0.9742,0.9330,0.9354,0.9622
0.1706,0.9306,0.9244,0.9738,0.9433,0.9476,0.9640
0.2166,0.9375,0.9332,0.9752,0.9433,0.9476,0.9627
0.2627,0.9444,0.9419,0.9762,0.9433,0.9476,0.9635
0.3087,0.9375,0.9362,0.9758,0.9433,0.9476,0.9634
0.3547,0.9375,0.9362,0.9762,0.9433,0.9476,0.9633
0.4008,0.9444,0.9449,0.9762,0.9381,0.9432,0.9630



Best by Val BAcc: 0.4008
Best by Test BAcc: 0.1706
